# Transfer Learning: бинарная классификация изображений

**ФИО студента:** Гришин Никита Андреевич

## Описание задания

**Цель работы:** самостоятельно собрать датасет для бинарной классификации изображений и исследовать эффективность различных предобученных нейросетевых архитектур с использованием Transfer Learning (Fine-tuning).

**Задачи:**
1. Собрать собственный набор изображений из двух классов и описать его состав.
2. Разделить данные на обучающую, валидационную и тестовую выборки.
3. Выбрать несколько архитектур с весами ImageNet (ResNet, EfficientNet, MobileNet, ConvNeXt).
4. Заменить у каждой модели классификационную голову на бинарную и выполнить Fine-tuning.
5. Оценить модели на тестовой выборке по метрикам Accuracy, Precision, Recall, F1-score.
6. Сравнить модели и сделать выводы.

## Датасет

Собрал датасет **Husky vs Malamute** — фотографии двух похожих северных пород собак:
- **husky** — сибирский хаски (класс 0)
- **malamute** — аляскинский маламут (класс 1)

Выбрал именно эти породы, потому что породы, в целом, похожи: окрас и общий вид у них очень похожи, отличаются они в основном размером и массивностью, формой морды, глазами и хвостом. Интересно посмотреть как модель научится различать мелкие детали, а не просто «собака / кошка»

- **Ссылка на датасет:** https://raw.githubusercontent.com/ITMO-NikitaGrishin/ITMO_ML/main/CV/laba_02/husky_vs_malamute.zip
- Архив (~49 МБ) лежит в репозитории рядом с блокнотом, внутри папки `husky/`, `malamute/` и файл `sources.csv` (класс, поисковый запрос, ссылка на картинку и на страницу-источник). Картинки уменьшены до 320 px по меньшей стороне, чтобы архив поместился в GitHub — на обучение это не влияет, перед подачей в модель они всё равно сжимаются до 256 px

Очистка датасета:
   - выкинул картинки, у которых в ссылке есть `vs`, `difference`, `compare`, `mix`, `wolf`, `cartoon`, `toy` и т.п., или упоминается другая порода — это чаще всего коллажи-сравнения, метисы и рисунки
   - выкинул картинки, на которых нет собаки: прогнал их через EfficientNet-B0 и оставил только те, где суммарная вероятность всех 118 «собачьих» классов ImageNet > 0.5
   - удалил дубликаты по перцептивному хешу (dHash, расстояние Хэмминга ≤ 6) — 44 шт.

**Итог:** 1230 изображений — 684 husky и 546 malamute

### Импорты

In [ ]:
import os
import time
import random
import zipfile
import hashlib
import urllib.request
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

### Загрузка датасета

Архив лежит в репозитории рядом с блокнотом; если его нет — скачиваю по прямой ссылке с GitHub. Проверяю SHA-256, чтобы данные были ровно те же, и распаковываю в `./data`

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/ITMO-NikitaGrishin/ITMO_ML/main/CV/laba_02/husky_vs_malamute.zip"
DATASET_SHA256 = "f04e6eea7844c8b3a7151447647068af026489a021fae5746dec417b22ac6a54"
ZIP_PATH = "husky_vs_malamute.zip"
DATA_DIR = "./data"
IMG_DIR = os.path.join(DATA_DIR, "husky_vs_malamute_320")

if not os.path.exists(ZIP_PATH):
    print("Скачиваю датасет:", DATASET_URL)
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)

with open(ZIP_PATH, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()
assert sha == DATASET_SHA256, f"SHA-256 архива не совпадает: {sha}"

if not os.path.exists(IMG_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(IMG_DIR)

CLASSES = ["husky", "malamute"]
paths, ys = [], []
for label, cls in enumerate(CLASSES):
    folder = os.path.join(IMG_DIR, cls)
    for f in sorted(os.listdir(folder)):
        paths.append(os.path.join(folder, f))
        ys.append(label)
paths, ys = np.array(paths), np.array(ys)

print("Всего изображений:", len(paths))
for label, cls in enumerate(CLASSES):
    print(f"  {cls:9s}: {(ys == label).sum()}")

### Состав датасета

Посмотрю на размеры изображений и на то, откуда они взяты

In [ ]:
sources = pd.read_csv(os.path.join(IMG_DIR, "sources.csv"))
sizes = np.array([Image.open(p).size for p in paths])  # (w, h)

info = pd.DataFrame({"label": [CLASSES[y] for y in ys], "w": sizes[:, 0], "h": sizes[:, 1]})
display(info.groupby("label").agg(count=("w", "size"), w_mean=("w", "mean"), h_mean=("h", "mean"),
                                  w_min=("w", "min"), h_min=("h", "min")).round(0))

sources["domain"] = sources["image_url"].str.extract(r"https?://([^/]+)/")[0].str.replace("www.", "", regex=False)
print("Уникальных сайтов-источников:", sources["domain"].nunique())
print("Самые частые источники:")
print(sources["domain"].value_counts().head(8).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
info["label"].value_counts().reindex(CLASSES).plot.bar(ax=axes[0], color=["tab:blue", "tab:orange"], rot=0)
axes[0].set_title("Количество изображений по классам")
for label, cls in enumerate(CLASSES):
    axes[1].scatter(sizes[ys == label, 0], sizes[ys == label, 1], s=6, alpha=0.5, label=cls)
axes[1].set_xlabel("ширина, px"); axes[1].set_ylabel("высота, px")
axes[1].set_title("Размеры изображений")
axes[1].legend()
plt.tight_layout()
plt.show()

### Примеры изображений

In [ ]:
rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for label, cls in enumerate(CLASSES):
    idx = rng.choice(np.where(ys == label)[0], 8, replace=False)
    for j, i in enumerate(idx):
        axes[label, j].imshow(Image.open(paths[i]).convert("RGB"))
        axes[label, j].set_title(cls)
        axes[label, j].axis("off")
plt.tight_layout()
plt.show()

### Разбиение на train / val / test

Делю со стратификацией по классу: 70% train, 15% val, 15% test. Val использую для выбора лучшей эпохи, test — только для итоговой оценки

In [ ]:
idx_all = np.arange(len(paths))
train_idx, tmp_idx = train_test_split(idx_all, test_size=0.30, stratify=ys, random_state=SEED)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.50, stratify=ys[tmp_idx], random_state=SEED)

split_info = pd.DataFrame({
    name: {cls: int((ys[idx] == k).sum()) for k, cls in enumerate(CLASSES)} | {"всего": len(idx)}
    for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]
})
split_info

### Предобработка и DataLoader'ы

Все модели предобучены на ImageNet, поэтому вход — 224×224 и нормализация средними / std ImageNet

Чтобы не декодировать большие JPEG на каждой эпохе, один раз загружаю все картинки в память, уменьшив меньшую сторону до 256 px

**Аугментации для train:**
- `RandomResizedCrop(224, scale=(0.6, 1.0))` — случайный кроп и масштаб
- `RandomHorizontalFlip` — собака, отражённая по горизонтали, остаётся той же породы
- `ColorJitter` — небольшие изменения яркости / контраста / насыщенности
- лёгкий поворот до ±10°

**Для val / test:** `Resize(256)` + `CenterCrop(224)`, без аугментаций

In [ ]:
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def load_resized(p, short=256):
    im = Image.open(p).convert("RGB")
    w, h = im.size
    s = short / min(w, h)
    return im.resize((max(short, round(w * s)), max(short, round(h * s))), Image.BICUBIC)

t0 = time.time()
images = [load_resized(p) for p in paths]
print(f"Загрузил {len(images)} изображений за {time.time() - t0:.1f} с")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class DogsDataset(Dataset):
    def __init__(self, idx, tf):
        self.idx = idx
        self.tf = tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        return self.tf(images[i]), int(ys[i])


BATCH_SIZE = 32
g = torch.Generator()
g.manual_seed(SEED)
train_loader = DataLoader(DogsDataset(train_idx, train_tf), batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader = DataLoader(DogsDataset(val_idx, eval_tf), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(DogsDataset(test_idx, eval_tf), batch_size=BATCH_SIZE, shuffle=False)

x, y = next(iter(train_loader))
print("batch:", x.shape, y.shape)

In [ ]:
# как выглядят аугментированные картинки из train
x, y = next(iter(DataLoader(DogsDataset(train_idx[:8], train_tf), batch_size=8)))
mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for j in range(8):
    axes[j].imshow((x[j] * std + mean).clamp(0, 1).permute(1, 2, 0))
    axes[j].set_title(CLASSES[y[j]])
    axes[j].axis("off")
plt.suptitle("Примеры после аугментаций (train)")
plt.tight_layout()
plt.show()

## Модели

Взял 4 архитектуры из `torchvision` с весами ImageNet-1k, специально разные по размеру и устройству:

| Модель | Тип | Особенности |
|---|---|---|
| **ResNet-50** | классическая CNN | residual-блоки, «стандартный» baseline для transfer learning |
| **EfficientNet-B0** | эффективная CNN | MBConv-блоки + squeeze-and-excitation, подобрана NAS-ом, мало параметров |
| **MobileNetV3-Large** | лёгкая CNN для мобильных устройств | depthwise-свёртки, h-swish, самая быстрая |
| **ConvNeXt-Tiny** | «современная» CNN | CNN, переделанная по образцу Vision Transformer (большие ядра 7×7, LayerNorm, GELU) |

У каждой модели заменяю последний линейный слой на `Linear(..., 2)` — два логита для классов husky / malamute, функция потерь — CrossEntropy

In [ ]:
def build_model(name):
    # возвращаю модель и её классификационную голову
    if name == "ResNet-50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, 2)
        head = m.fc
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
        head = m.classifier
    elif name == "MobileNetV3-Large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
        head = m.classifier
    elif name == "ConvNeXt-Tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
        head = m.classifier
    else:
        raise ValueError(name)
    return m, head


MODEL_NAMES = ["ResNet-50", "EfficientNet-B0", "MobileNetV3-Large", "ConvNeXt-Tiny"]

n_params = {}
for name in MODEL_NAMES:
    m, head = build_model(name)
    n_params[name] = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name:18s}: {n_params[name]:5.1f} M параметров, голова: {head}")
    del m

## Fine-tuning

Обучаю в два этапа — это стандартная схема transfer learning:

1. **Обучение только головы** (`HEAD_EPOCHS` эпох). Веса backbone заморожены, учится только новый линейный слой. Новый слой инициализирован случайно, и если сразу размораживать всю сеть, большие градиенты от «случайной» головы могут испортить предобученные признаки
2. **Fine-tuning всей сети** (`FT_EPOCHS` эпох). Размораживаю все слои и дообучаю с маленьким learning rate для backbone и в 10 раз большим для головы, lr уменьшается по косинусу

После каждой эпохи считаю loss, Accuracy и F1 на валидации. Сохраняю веса эпохи с лучшим val F1 (при равенстве — с меньшим val loss), их потом и оцениваю на тесте

Дополнительно после первого этапа оцениваю модель на тесте — так будет видно, сколько даёт fine-tuning всей сети по сравнению с обучением только головы (linear probing)

**Параметры обучения** (одинаковые для всех моделей):

In [ ]:
CFG = dict(
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    head_epochs=3,          # этап 1: только голова
    head_lr=1e-3,
    ft_epochs=12,           # этап 2: вся сеть
    ft_lr_backbone=1e-4,
    ft_lr_head=1e-3,
    weight_decay=1e-4,
    optimizer="AdamW",
    scheduler="CosineAnnealingLR (этап 2)",
    label_smoothing=0.1,
    seed=SEED,
)
pd.Series(CFG, name="значение").to_frame()

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


def run_epoch(model, loader, optimizer=None):
    # одна эпоха обучения (если передан optimizer) или оценки
    train = optimizer is not None
    model.train(train)
    total_loss, all_y, all_p = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(x)
            all_y.append(y.cpu().numpy())
            all_p.append(logits.argmax(1).cpu().numpy())
    y_true, y_pred = np.concatenate(all_y), np.concatenate(all_p)
    return total_loss / len(loader.dataset), accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, zero_division=0)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for x, _ in loader:
        probs.append(torch.softmax(model(x.to(device)), 1)[:, 1].cpu().numpy())
    return np.concatenate(probs)


def test_metrics(y_true, prob, thr=0.5):
    y_pred = (prob >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0)}


def fine_tune(name):
    set_seed(SEED)
    model, head = build_model(name)
    model = model.to(device)
    head_ids = {id(p) for p in head.parameters()}
    history = []

    def log(stage, epoch, tr, va):
        history.append(dict(stage=stage, epoch=epoch, train_loss=tr[0], train_acc=tr[1],
                            val_loss=va[0], val_acc=va[1], val_f1=va[2]))
        print(f"  [{stage}] эпоха {epoch:2d}: train loss {tr[0]:.4f} acc {tr[1]:.3f} | "
              f"val loss {va[0]:.4f} acc {va[1]:.3f} F1 {va[2]:.3f}")

    t0 = time.time()
    # этап 1: замораживаю всё, кроме головы
    for p in model.parameters():
        p.requires_grad = id(p) in head_ids
    opt = torch.optim.AdamW(head.parameters(), lr=CFG["head_lr"], weight_decay=CFG["weight_decay"])
    for ep in range(1, CFG["head_epochs"] + 1):
        tr = run_epoch(model, train_loader, opt)
        log("head", ep, tr, run_epoch(model, val_loader))
    head_only = test_metrics(ys[test_idx], predict(model, test_loader))

    # этап 2: размораживаю всю сеть
    for p in model.parameters():
        p.requires_grad = True
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": backbone_params, "lr": CFG["ft_lr_backbone"]},
                             {"params": head.parameters(), "lr": CFG["ft_lr_head"]}],
                            weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["ft_epochs"])
    best_key, best_state, best_epoch = (-1, 0), None, None
    for ep in range(1, CFG["ft_epochs"] + 1):
        tr = run_epoch(model, train_loader, opt)
        sched.step()
        va = run_epoch(model, val_loader)
        log("full", ep, tr, va)
        key = (va[2], -va[0])
        if key > best_key:
            best_key, best_state, best_epoch = key, copy.deepcopy(model.state_dict()), ep
    train_time = time.time() - t0

    model.load_state_dict(best_state)
    os.makedirs("weights", exist_ok=True)
    torch.save(best_state, os.path.join("weights", name.replace(" ", "_") + ".pt"))
    print(f"  лучшая эпоха fine-tuning: {best_epoch}, val F1 = {best_key[0]:.4f}, время обучения {train_time:.0f} с")
    return model, pd.DataFrame(history), head_only, train_time, best_epoch

### Обучение моделей

In [ ]:
results = {}
for name in MODEL_NAMES:
    print(f"===== {name} =====")
    model, hist, head_only, train_time, best_epoch = fine_tune(name)
    results[name] = dict(model=model, history=hist, head_only=head_only,
                         train_time=train_time, best_epoch=best_epoch)
    torch.cuda.empty_cache()

### Кривые обучения

In [ ]:
fig, axes = plt.subplots(2, len(MODEL_NAMES), figsize=(5 * len(MODEL_NAMES), 8), sharey="row")
for j, name in enumerate(MODEL_NAMES):
    h = results[name]["history"]
    x = np.arange(1, len(h) + 1)
    axes[0, j].plot(x, h["train_loss"], label="train")
    axes[0, j].plot(x, h["val_loss"], label="val")
    axes[1, j].plot(x, h["train_acc"], label="train acc")
    axes[1, j].plot(x, h["val_acc"], label="val acc")
    axes[1, j].plot(x, h["val_f1"], "--", label="val F1")
    for ax in axes[:, j]:
        ax.axvline(CFG["head_epochs"] + 0.5, color="gray", ls=":", lw=1)
        ax.axvline(CFG["head_epochs"] + results[name]["best_epoch"], color="red", ls=":", lw=1)
        ax.grid(alpha=0.3)
        ax.set_xlabel("эпоха")
    axes[0, j].set_title(name)
    axes[0, j].legend(); axes[1, j].legend(loc="lower right")
axes[0, 0].set_ylabel("Loss")
axes[1, 0].set_ylabel("Accuracy / F1")
fig.suptitle("Серая линия — начало fine-tuning всей сети, красная — выбранная эпоха", fontsize=13)
plt.tight_layout()
plt.show()

**Комментарий по обучению:**
- _заполнить после запуска: как ведут себя loss / accuracy на этапе «только голова» и после разморозки всей сети, есть ли переобучение, на какой эпохе лучший val F1_

## Оценка на тестовой выборке

Для каждой модели (веса лучшей по val F1 эпохи) считаю на test Accuracy, Precision, Recall и F1 (положительный класс — malamute), а также время инференса одного изображения

In [ ]:
@torch.no_grad()
def inference_time(model, n_runs=50):
    # время на одно изображение (batch = 1), после прогрева
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(10):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_runs):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n_runs * 1000


y_test = ys[test_idx]
rows, head_rows = [], []
for name in MODEL_NAMES:
    r = results[name]
    r["test_prob"] = predict(r["model"], test_loader)
    m = test_metrics(y_test, r["test_prob"])
    r["test_metrics"] = m
    rows.append({"Модель": name, **m,
                 "Параметры, M": n_params[name],
                 "Время обучения, с": r["train_time"],
                 "Инференс, мс/изобр.": inference_time(r["model"])})
    head_rows.append({"Модель": name, "F1 (только голова)": r["head_only"]["F1"],
                      "F1 (fine-tuning)": m["F1"],
                      "Accuracy (только голова)": r["head_only"]["Accuracy"],
                      "Accuracy (fine-tuning)": m["Accuracy"]})

summary = pd.DataFrame(rows).set_index("Модель")
summary.round(4)

### Матрицы ошибок

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.5 * len(MODEL_NAMES), 4))
for ax, name in zip(axes, MODEL_NAMES):
    y_pred = (results[name]["test_prob"] >= 0.5).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=CLASSES).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nF1 = {results[name]['test_metrics']['F1']:.3f}")
plt.tight_layout()
plt.show()

## Сравнение моделей

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1"]
ax = summary[metrics].plot.bar(figsize=(11, 5), rot=0, width=0.8)
ax.set_ylim(max(0, summary[metrics].values.min() - 0.1), 1.0)
ax.set_title("Метрики на тестовой выборке")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=7, rotation=90, padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# качество vs размер и скорость модели
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in MODEL_NAMES:
    s = summary.loc[name]
    axes[0].scatter(s["Параметры, M"], s["F1"], s=80)
    axes[0].annotate(name, (s["Параметры, M"], s["F1"]), textcoords="offset points", xytext=(6, 4))
    axes[1].scatter(s["Инференс, мс/изобр."], s["F1"], s=80)
    axes[1].annotate(name, (s["Инференс, мс/изобр."], s["F1"]), textcoords="offset points", xytext=(6, 4))
axes[0].set_xlabel("Параметров, млн"); axes[1].set_xlabel(f"Инференс на {device.type.upper()}, мс / изображение")
for ax in axes:
    ax.set_ylabel("F1 на test")
    ax.grid(alpha=0.3)
axes[0].set_title("F1 vs размер модели")
axes[1].set_title("F1 vs скорость")
plt.tight_layout()
plt.show()

### Только голова vs Fine-tuning всей сети

Сравниваю качество на test после первого этапа (обучена только новая голова, backbone заморожен) и после полного fine-tuning

In [ ]:
head_df = pd.DataFrame(head_rows).set_index("Модель")
head_df["Прирост F1"] = head_df["F1 (fine-tuning)"] - head_df["F1 (только голова)"]
head_df.round(4)

**Комментарий по сравнению:**
- _заполнить после запуска: какая модель лучшая по F1, разница между моделями, баланс Precision / Recall, соотношение качества с размером и скоростью, прирост от fine-tuning всей сети относительно обучения только головы_

## Анализ ошибок

Посмотрю, на каких тестовых изображениях ошибаются модели и совпадают ли эти ошибки у разных архитектур

In [ ]:
pred_df = pd.DataFrame({name: (results[name]["test_prob"] >= 0.5).astype(int) for name in MODEL_NAMES})
wrong = pred_df.ne(y_test, axis=0)
n_wrong = wrong.sum(axis=1)
print("Число тестовых изображений, на которых ошибаются k моделей:")
print(n_wrong.value_counts().sort_index().rename_axis("k").to_string())

# ансамбль: усредняю вероятности всех моделей
ens_prob = np.mean([results[name]["test_prob"] for name in MODEL_NAMES], axis=0)
ens = test_metrics(y_test, ens_prob)
print("\nАнсамбль (среднее вероятностей 4 моделей):", {k: round(v, 4) for k, v in ens.items()})

In [ ]:
# изображения, на которых ошибается больше всего моделей
order = np.argsort(-n_wrong.values, kind="stable")
order = [k for k in order if n_wrong.iloc[k] > 0][:12]
if order:
    cols = 6
    rows_n = (len(order) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.3 * cols, 3.8 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, k in zip(axes.flat, order):
        i = test_idx[k]
        ax.imshow(images[i])
        probs = " ".join(f"{results[n]['test_prob'][k]:.2f}" for n in MODEL_NAMES)
        ax.set_title(f"истина: {CLASSES[ys[i]]}\nошиблись {n_wrong.iloc[k]}/4\np(mal): {probs}", fontsize=9)
    fig.suptitle("Самые трудные тестовые изображения (p(mal) в порядке: " + ", ".join(MODEL_NAMES) + ")")
    plt.tight_layout()
    plt.show()

**Комментарий по ошибкам:**
- _заполнить после запуска: на каких изображениях ошибаются модели (щенки, плохой ракурс, несколько собак, ошибки разметки), совпадают ли ошибки у разных архитектур, помогает ли ансамбль_

## Выводы

1. **Датасет.** Собран собственный датасет Husky vs Malamute: 1230 изображений (684 husky, 546 malamute), разделён стратифицированно на train / val / test = 70 / 15 / 15

2. **Итоговое сравнение на тестовой выборке:**

| Модель | Accuracy | Precision | Recall | F1 | Параметры, M | Инференс, мс |
|---|---|---|---|---|---|---|
| ResNet-50 | | | | | 23.5 | |
| EfficientNet-B0 | | | | | 4.0 | |
| MobileNetV3-Large | | | | | 4.2 | |
| ConvNeXt-Tiny | | | | | 27.8 | |

3. **Fine-tuning vs только голова.** _заполнить_

4. **Ошибки.** _заполнить_

5. **Общий вывод.** _заполнить_